[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/01/%EB%A0%88%EC%8A%A8%2001%20%E2%80%94%20%EC%9B%B9%20%EC%9E%90%EB%8F%99%ED%99%94%20%EA%B0%9C%EC%9A%94%20%2B%20HTTP%20%2B%20BeautifulSoup%20%EC%9E%85%EB%AC%B8.ipynb)

# 레슨 01 — 웹 자동화 개요 + HTTP + BeautifulSoup 입문

> 교사용 통합 노트북입니다. 학생에게는 학생용 노트북만 공유합니다.

In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/01/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', text))

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 4. HTML 파일 읽기

첫 샘플은 mini_shop.html이다. 실제 쇼핑몰이 아니라 수업용으로 만든 합성 HTML이다.

In [ ]:
html = load_text('mini_shop.html')
print(type(html))
print(len(html))
print(html[:300])


HTML은 그냥 긴 문자열이다. 문자열 상태에서는 태그를 찾기 어렵기 때문에 BeautifulSoup 객체로 바꾼다.

In [ ]:
soup = BeautifulSoup(html, 'html.parser')
print(soup.title.text.strip())
print(soup.select_one('h1').text.strip())


---

## 5. CSS selector로 원하는 요소 고르기

웹 자동화에서 가장 많이 쓰는 문법은 CSS selector다.

- 'h1': h1 태그 선택
- '.product-card': class가 product-card인 요소 선택
- '#main': id가 main인 요소 선택
- 'article.product-card': article 태그이면서 product-card 클래스인 요소 선택
- '.product-card .name': product-card 안쪽의 name 클래스 선택

In [ ]:
cards = soup.select('.product-card')
print('상품 카드 수:', len(cards))

first = cards[0]
print(first.select_one('.name').text.strip())
print(first.select_one('.price').text.strip())
print(first['data-category'])


---

## 6. 텍스트 정리와 숫자 변환

웹에서 가져온 값은 대부분 문자열이다. 가격처럼 쉼표와 원 문자가 섞인 문자열은 숫자로 바꿔야 비교와 정렬이 가능하다.

In [ ]:
price_text = first.select_one('.price').text.strip()
price = clean_int(price_text)
print(price_text, '->', price)


---

## 7. 여러 상품을 표 형태로 만들기

반복문으로 각 상품 카드에서 같은 위치의 값을 꺼낸다. 리스트 안에 딕셔너리를 쌓으면 표처럼 다루기 쉽다.

In [ ]:
products = []
for card in cards:
    item = {
        'name': card.select_one('.name').text.strip(),
        'category': card['data-category'],
        'price': clean_int(card.select_one('.price').text),
        'rating': float(card.select_one('.rating').text.replace('★', '').strip()),
        'stock': clean_int(card.select_one('.stock').text),
        'detail_url': urljoin('https://example.com', card.select_one('a.detail')['href']),
    }
    products.append(item)

print(products[0])
print('총 상품:', len(products))


---

## 8. 조건으로 걸러 보기

리스트 컴프리헨션을 쓰면 원하는 조건의 데이터만 뽑을 수 있다.

In [ ]:
expensive = [p for p in products if p['price'] >= 1500000]
high_rating = [p for p in products if p['rating'] >= 4.7]
out_of_stock = [p for p in products if p['stock'] == 0]

print('150만원 이상:', len(expensive))
print('평점 4.7 이상:', len(high_rating))
print('품절:', len(out_of_stock))


---

## 9. 공지 페이지 파싱

두 번째 샘플은 notices.html이다. 공지 목록처럼 행이 반복되는 구조를 연습한다.

In [ ]:
notice_html = load_text('notices.html')
notice_soup = BeautifulSoup(notice_html, 'html.parser')
rows = notice_soup.select('li.notice-item')
print('공지 수:', len(rows))

first_notice = rows[0]
print(first_notice.select_one('.date').text.strip())
print(first_notice.select_one('.title').text.strip())
print(first_notice.select_one('.department').text.strip())


---

## 10. CSV로 저장하기

수집 결과는 화면에 print만 하지 말고 파일로 남겨야 한다. 이번 레슨에서는 csv 모듈을 쓴다.

In [ ]:
output_path = 'lesson01_products.csv'
with open(output_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'category', 'price', 'rating', 'stock', 'detail_url'])
    writer.writeheader()
    writer.writerows(products)

print('saved:', output_path)


---

## 11. 현업 활용 사례

웹 자동화는 데이터 분석보다 먼저 쓰이는 경우가 많다. 분석할 데이터가 파일로 주어지지 않을 때, 웹 페이지의 공지·가격·리뷰·채용공고·자료실 링크를 정리해 데이터셋을 만든다.

예를 들어 가격 비교 서비스는 상품명, 가격, 재고, 판매처 링크를 주기적으로 확인한다. 다만 실서비스는 단순 requests만 쓰지 않는다. robots.txt, 약관, 요청 제한, IP 차단 방지, 중복 저장 방지, 변경 이력 관리까지 운영 규칙이 필요하다.

1강에서는 운영 전체가 아니라 '한 페이지를 안전하게 읽고 표로 바꾸는 기본기'만 익힌다.

## 12. HTTP 응답을 안전하게 확인하는 법

웹 자동화에서 가장 흔한 실수는 "HTML이 왔겠지"라고 가정하고 바로 파싱하는 것이다. 실제 사이트에서는 URL 오타, 권한 없음, 서버 오류, 너무 빠른 요청 때문에 HTML 대신 에러 페이지가 올 수 있다. 그래서 요청 결과를 파싱하기 전에 응답 상태를 확인해야 한다.

> **🛡️ 웹 자동화 안전 한스푼 — HTTP 요청/응답과 status_code**
>
> - **뜻**: 요청은 클라이언트가 서버에 보내는 질문이고, 응답은 서버가 돌려주는 결과다. `status_code` 는 그 결과의 상태 번호다.
> - **왜 중요한가**: 200이 아니면 원하는 HTML이 아닐 수 있다. 404는 주소 없음, 403은 거부, 429는 너무 많은 요청을 뜻할 수 있다.
> - **수업 기준**: 외부 요청 코드는 항상 `timeout` 과 상태 확인을 같이 둔다.
> - **실수 예시**: `requests.get(url).text` 만 쓰고 상태를 확인하지 않은 채 BeautifulSoup에 넘긴다.

In [ ]:
sample_url = DATA_BASE + '/mini_shop.html' if DATA_BASE.startswith('http') else None

if sample_url:
    response = requests.get(sample_url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
    print('status:', response.status_code)
    response.raise_for_status()
    print(response.text[:80])
else:
    print('로컬 실행 중이므로 외부 HTTP 요청 예시는 건너뜁니다.')


> **🛡️ 웹 자동화 안전 한스푼 — timeout**
>
> - **뜻**: 서버가 일정 시간 안에 응답하지 않으면 기다리기를 멈추는 제한 시간이다.
> - **왜 중요한가**: timeout이 없으면 네트워크 문제 하나로 노트북 셀이 계속 멈춰 있을 수 있다.
> - **수업 기준**: `requests.get(url, timeout=10)` 을 기본으로 쓴다.
> - **실수 예시**: 여러 URL을 반복 요청하면서 timeout을 넣지 않아 수업 전체가 멈춘다.

`raise_for_status()` 는 400번대/500번대 응답을 그냥 넘어가지 않게 해 준다. 처음에는 에러가 뜨는 것이 불편해 보이지만, 조용히 틀린 HTML을 파싱하는 것보다 훨씬 안전하다.

---

## 13. selector 디버깅 루틴

selector가 틀리면 BeautifulSoup은 에러를 내지 않고 빈 리스트를 돌려주는 경우가 많다. 그래서 한 번에 긴 코드를 쓰지 말고 다음 순서로 확인한다.

1. 문서 전체가 제대로 읽혔는지 `len(html)` 을 본다.
2. `soup.title` 또는 `h1` 처럼 확실한 요소를 먼저 찾는다.
3. 반복 단위 selector의 개수를 `len(soup.select(...))` 로 본다.
4. 첫 번째 반복 단위 하나에서 내부 요소를 확인한다.
5. 그 다음에야 반복문으로 전체를 처리한다.

In [ ]:
debug_cards = soup.select('.product-card')
print('cards:', len(debug_cards))

if debug_cards:
    sample = debug_cards[0]
    for selector in ['.name', '.price', '.rating', '.stock', 'a.detail']:
        found = sample.select_one(selector)
        print(selector, '=>', found.text.strip() if found else '없음')


> **🛡️ 웹 자동화 안전 한스푼 — HTML selector**
>
> - **뜻**: HTML에서 원하는 태그를 고르는 주소 같은 표현이다. `.class`, `#id`, `tag[attr=value]` 를 조합한다.
> - **왜 중요한가**: selector가 불안정하면 사이트 구조가 조금만 바뀌어도 자동화가 깨진다.
> - **수업 기준**: 반복 단위 selector를 먼저 고르고, 내부 selector는 그 안에서 다시 찾는다.
> - **실수 예시**: 전체 문서에서 `.price` 를 한 번만 찾아 첫 상품 가격만 계속 저장한다.

selector가 비어 있으면 HTML을 다시 보는 것이 먼저다. 코드를 더 복잡하게 만들기 전에 "내가 고르려는 반복 단위가 실제로 어떤 태그와 class를 갖는가"를 눈으로 확인한다.

---

## 14. 상대 URL과 절대 URL

HTML 안의 링크는 `/products/air-13` 처럼 상대 경로로 들어 있는 경우가 많다. 사람이 브라우저로 볼 때는 현재 사이트 주소가 자동으로 붙지만, CSV로 저장할 때는 기준 주소를 붙여 절대 URL로 바꿔야 한다.

In [ ]:
relative_href = first.select_one('a.detail')['href']
absolute_href = urljoin('https://example.com/shop/', relative_href)
print(relative_href)
print(absolute_href)


상대 URL을 그대로 저장하면 나중에 CSV만 열었을 때 링크가 어디를 가리키는지 알기 어렵다. `urljoin` 은 기준 주소와 상대 경로를 안전하게 합쳐 준다.

> **🛡️ 웹 자동화 안전 한스푼 — relative URL / absolute URL**
>
> - **뜻**: 상대 URL은 현재 사이트를 기준으로 한 짧은 주소이고, 절대 URL은 `https://...` 로 시작하는 완전한 주소다.
> - **왜 중요한가**: 수집 결과를 CSV로 저장하면 기준 사이트 정보가 사라질 수 있다.
> - **수업 기준**: 링크를 저장할 때는 `urljoin` 으로 절대 URL 형태를 만든다.
> - **실수 예시**: `/notice/12` 만 저장해서 나중에 어느 사이트의 공지인지 알 수 없다.

---

## 15. robots.txt와 요청 간격

이번 레슨은 합성 HTML 파일을 쓰기 때문에 실제 서버에 반복 요청하지 않는다. 그래도 처음부터 robots.txt와 요청 간격을 말하는 이유는 습관 때문이다. 자동화 코드는 반복문을 쓰는 순간 사람이 클릭하는 속도보다 훨씬 빨라질 수 있다.

> **🛡️ 웹 자동화 안전 한스푼 — robots.txt**
>
> - **뜻**: 사이트가 자동화 프로그램에게 어느 경로를 허용하거나 제한하는지 알려주는 참고 파일이다.
> - **왜 중요한가**: 공개 페이지라도 운영자가 자동 접근을 제한하고 싶을 수 있다.
> - **수업 기준**: 실제 사이트 예제는 robots와 약관을 확인한 뒤 강사가 지정한 범위에서만 요청한다.
> - **실수 예시**: "브라우저에서 보이니까 마음대로 반복 요청해도 된다"고 생각한다.

> **🛡️ 웹 자동화 안전 한스푼 — 요청 간격(rate limit)**
>
> - **뜻**: 여러 페이지를 요청할 때 요청 사이에 쉬는 시간을 두는 규칙이다.
> - **왜 중요한가**: 너무 빠른 반복 요청은 서버 부하나 차단의 원인이 된다.
> - **수업 기준**: 실제 외부 요청 반복문에는 최소 `time.sleep(1)` 을 둔다.
> - **실수 예시**: 1초에 수십 번씩 URL을 바꿔 요청한다.

In [ ]:
targets_text = load_text('targets.csv')
print(targets_text.splitlines()[:4])

# 실제 사이트 요청 예시는 아니다. 반복 처리 구조만 보여준다.
for index, filename in enumerate(['mini_shop.html', 'notices.html'], start=1):
    text = load_text(filename)
    print(index, filename, len(text))
    time.sleep(0.2)  # 수업 fixture라 짧게 둔다. 실제 외부 사이트는 더 길게 둔다.


---

## 16. 실무에서는 무엇을 더 추가하나

실무 자동화는 1강 코드보다 더 많은 보호 장치를 둔다.

| 항목 | 1강에서 하는 것 | 실무에서 추가하는 것 |
|---|---|---|
| 요청 | fixture 또는 raw 파일 읽기 | 재시도, 요청 간격, 실패 로그 |
| 파싱 | selector로 텍스트 추출 | selector 변경 감지, 누락 필드 알림 |
| 저장 | CSV 1개 저장 | 날짜별 파일명, 중복 제거, DB 저장 |
| 검증 | 개수와 일부 값 출력 | 스키마 검증, null 비율, 이전 결과와 비교 |
| 운영 | 노트북 수동 실행 | 스케줄러, 알림, 장애 대응 |

이 표를 보면 1강의 목적이 분명해진다. 오늘은 실무 전체를 다 만들지 않는다. 대신 반복 단위를 고르고, 내부 값을 읽고, 숫자로 바꾸고, CSV로 저장하는 기초 루틴을 정확히 익힌다. 이 루틴이 흔들리면 페이지네이션, 브라우저 자동화, 재시도 같은 뒤 레슨도 모두 흔들린다.

---

## 17. 제출 전 마무리 체크

이번 레슨을 마치기 전에 아래 질문에 답해 본다.

- HTML 문자열을 BeautifulSoup 객체로 바꾸는 이유를 말할 수 있는가?
- `select` 와 `select_one` 의 차이를 말할 수 있는가?
- class selector 앞에 `.` 을 붙이는 이유를 설명할 수 있는가?
- 가격/조회수 문자열을 숫자로 바꾸지 않으면 어떤 문제가 생기는가?
- 상대 URL을 절대 URL로 바꾸는 이유를 말할 수 있는가?
- 실제 사이트에서 같은 코드를 실행하기 전에 robots.txt, 약관, 요청 간격을 확인해야 하는 이유를 말할 수 있는가?

## 데이터 출처와 안전 규칙

이 레슨의 `mini_shop.html`, `notices.html`, `targets.csv`, `robots_sample.txt` 는 수업용 합성 데이터다. 실존 사이트나 개인 정보를 포함하지 않는다. `requests` 예제는 raw GitHub의 수업 파일 또는 로컬 fixture만 읽도록 설계되어 있다. 학생은 강사가 별도로 허가하지 않은 실제 사이트에 반복 요청을 보내지 않는다.

# 레슨 01 — 실습 문제 정답지

> 🔒 교사·관리자 전용. 학생에게 배포 금지.

각 문제의 모범 답안과 채점 포인트를 정리했다. 학생 답안은 출력값만 보지 말고 selector가 안정적인지, 텍스트 정리와 타입 변환이 정확한지 확인한다.

## 0. 환경 셀

In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/01/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', text))

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 문제 1 정답 — 환경 셀 실행과 파일 목록 확인

In [ ]:
shop_html = load_text('mini_shop.html')
notice_html = load_text('notices.html')
target_csv = load_text('targets.csv')
print('shop:', len(shop_html))
print('notice:', len(notice_html))
print('targets:', len(target_csv))


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 2 정답 — HTML 제목과 h1 찾기

In [ ]:
soup = BeautifulSoup(shop_html, 'html.parser')
page_title = soup.title.text.strip()
main_title = soup.select_one('h1').text.strip()
print(page_title)
print(main_title)


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 3 정답 — 상품 카드 개수 세기

In [ ]:
cards = soup.select('.product-card')
print('상품 카드 수:', len(cards))


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 4 정답 — 첫 번째 상품 정보 읽기

In [ ]:
first = cards[0]
name = first.select_one('.name').text.strip()
category = first['data-category']
price_text = first.select_one('.price').text.strip()
href = first.select_one('a.detail')['href']
print(name, category, price_text, href)


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 5 정답 — 가격 문자열을 숫자로 바꾸기

In [ ]:
price_text = first.select_one('.price').text
price = clean_int(price_text)
print(price, type(price))


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 6 정답 — 모든 상품을 딕셔너리 리스트로 만들기

In [ ]:
products = []
for card in cards:
    item = {
        'name': card.select_one('.name').text.strip(),
        'category': card['data-category'],
        'price': clean_int(card.select_one('.price').text),
        'rating': float(card.select_one('.rating').text.replace('★', '').strip()),
        'stock': clean_int(card.select_one('.stock').text),
        'detail_url': urljoin('https://example.com', card.select_one('a.detail')['href']),
    }
    products.append(item)
print(products[0])
print(len(products))


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 7 정답 — 카테고리별 상품 수 세기

In [ ]:
counts = {}
for item in products:
    key = item['category']
    counts[key] = counts.get(key, 0) + 1
print(counts)


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 8 정답 — 150만원 이상 상품 찾기

In [ ]:
expensive = [p for p in products if p['price'] >= 1500000]
for p in expensive:
    print(p['name'], p['price'])


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 9 정답 — 평점 4.7 이상 상품 찾기

In [ ]:
top_rated = [p for p in products if p['rating'] >= 4.7]
for p in top_rated:
    print(p['name'], p['rating'], p['stock'])


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 10 정답 — 품절 상품 찾기

In [ ]:
sold_out = [p for p in products if p['stock'] == 0]
for p in sold_out:
    print(p['name'], p['detail_url'])


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 11 정답 — 공지 목록 파싱하기

In [ ]:
notice_soup = BeautifulSoup(notice_html, 'html.parser')
notice_rows = notice_soup.select('li.notice-item')
notices = []
for row in notice_rows:
    notices.append({
        'date': row.select_one('.date').text.strip(),
        'title': row.select_one('.title').text.strip(),
        'department': row.select_one('.department').text.strip(),
        'views': clean_int(row.select_one('.views').text),
    })
print(notices[0])
print(len(notices))


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 12 정답 — 운영팀 공지만 필터링하기

In [ ]:
ops_notices = [n for n in notices if n['department'] == '운영팀']
for n in ops_notices:
    print(n['date'], n['title'])


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 13 정답 — 조회수 상위 공지 3개 찾기

In [ ]:
top3 = sorted(notices, key=lambda n: n['views'], reverse=True)[:3]
for n in top3:
    print(n['title'], n['views'])


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 14 정답 — targets.csv에서 허용 대상만 읽기

In [ ]:
import io
reader = csv.DictReader(io.StringIO(target_csv))
allowed = []
for row in reader:
    if row['allowed'] == 'yes':
        allowed.append(row)
for row in allowed:
    print(row['filename'], row['purpose'])


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 문제 15 정답 — 수집 요약 문장 만들기

In [ ]:
summary1 = f"총 상품 수는 {len(products)}개입니다."
summary2 = f"품절 상품은 {len(sold_out)}개입니다."
summary3 = f"조회수 1위 공지는 '{top3[0]['title']}'입니다."
print(summary1)
print(summary2)
print(summary3)


### 왜 이 코드가 정답인지

이 코드는 문제에서 요구한 HTML 또는 CSV 원본을 먼저 올바른 자료구조로 바꾼 뒤, 필요한 위치만 선택한다. BeautifulSoup을 사용할 때는 태그 이름보다 class 기반 selector를 쓰는 편이 수업용 HTML 구조와 잘 맞고, 텍스트는 strip으로 공백을 제거한다. 가격·재고·조회수처럼 비교나 정렬이 필요한 값은 clean_int로 숫자만 남겨 int로 바꾼다. 이 변환이 빠지면 문자열 정렬이나 문자열 비교가 되어 잘못된 결과가 나온다.

### 채점 포인트

- selector가 실제 HTML 구조와 맞는가.
- text와 attribute를 구분해서 읽었는가.
- 숫자 필드는 int 또는 float로 변환했는가.
- 반복 추출 문제는 리스트 길이가 원본 카드/행 개수와 일치하는가.

### 자주 보이는 오답

- select_one 결과에 바로 인덱스를 붙이거나, select 결과 리스트에서 text를 바로 읽으려는 실수.
- 가격 문자열을 숫자로 바꾸지 않고 그대로 비교하는 실수.
- href를 상대 경로 그대로 저장해 나중에 열 수 없는 링크가 되는 실수.

---

## 채점 시 우선 확인 순서

채점은 문제 번호 순서대로 하되, 실제로는 의존 관계를 먼저 본다. 문제 1~3에서 환경과 반복 단위가 맞지 않으면 뒤 문제 대부분이 연쇄적으로 실패한다. 이 경우 뒤 문제를 하나씩 고치게 하기보다 `shop_html`, `soup`, `cards` 가 만들어지는 흐름을 먼저 복구하게 한다.

| 확인 순서 | 봐야 할 것 | 통과 기준 |
|---|---|---|
| 1 | 환경 셀 | `DATA_BASE`, `load_text`, `clean_int` 가 정상 정의됨 |
| 2 | HTML 로드 | `shop_html`, `notice_html`, `target_csv` 가 비어 있지 않음 |
| 3 | 반복 단위 | `.product-card`, `li.notice-item` 개수가 원본과 일치 |
| 4 | 타입 변환 | 가격/재고/조회수가 숫자로 변환됨 |
| 5 | 저장 | `lesson01_products.csv` 가 헤더와 행을 포함 |
| 6 | 요약 | 마지막 문장이 실제 출력 숫자와 충돌하지 않음 |

## 부분 점수 안내

학생이 모범 답안과 다른 코드를 작성해도 같은 구조를 만족하면 통과시킬 수 있다. 예를 들어 카테고리별 집계를 `collections.Counter` 로 풀어도 좋고, CSV 읽기를 `pandas.read_csv` 로 시도한 학생도 결과가 맞으면 인정할 수 있다. 다만 1강의 학습 목표가 BeautifulSoup selector와 표준 라이브러리 `csv` 흐름이므로, 다른 도구로 풀었더라도 수업 기준 풀이를 한 번 더 설명하게 한다.

감점 또는 보완 대상은 다음과 같다.

- selector 결과가 `None` 인데 `.text` 를 바로 호출해 노트북이 중간에 멈춘다.
- 가격을 문자열 상태로 비교해 `"900000"` 과 `"1500000"` 의 순서가 잘못된다.
- 상대 URL을 그대로 저장해 CSV만 봤을 때 링크의 기준 사이트를 알 수 없다.
- 실제 사이트에 바로 적용하겠다고 쓰면서 robots.txt, 약관, 요청 간격 언급이 없다.
- 결과 요약 문장이 코드 출력과 맞지 않는다.

# 레슨 01 — 최종 미션 모범 답안

> 🔒 교사·관리자 전용. 학생에게 배포 금지.

## 환경 셀

In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/01/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', text))

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


## 모범 코드

In [ ]:
shop_html = load_text('mini_shop.html')
notice_html = load_text('notices.html')
shop_soup = BeautifulSoup(shop_html, 'html.parser')
notice_soup = BeautifulSoup(notice_html, 'html.parser')

products = []
for card in shop_soup.select('.product-card'):
    products.append({
        'name': card.select_one('.name').text.strip(),
        'category': card['data-category'],
        'price': clean_int(card.select_one('.price').text),
        'rating': float(card.select_one('.rating').text.replace('★', '').strip()),
        'stock': clean_int(card.select_one('.stock').text),
        'detail_url': urljoin('https://example.com', card.select_one('a.detail')['href']),
    })

notices = []
for row in notice_soup.select('li.notice-item'):
    notices.append({
        'date': row.select_one('.date').text.strip(),
        'title': row.select_one('.title').text.strip(),
        'department': row.select_one('.department').text.strip(),
        'views': clean_int(row.select_one('.views').text),
    })

with open('lesson01_products.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'category', 'price', 'rating', 'stock', 'detail_url'])
    writer.writeheader()
    writer.writerows(products)

sold_out = [p for p in products if p['stock'] == 0]
top_rated = [p for p in products if p['rating'] >= 4.7]
top_notices = sorted(notices, key=lambda n: n['views'], reverse=True)[:3]

print('상품 수:', len(products))
print('공지 수:', len(notices))
print('품절:', [p['name'] for p in sold_out])
print('고평점:', [p['name'] for p in top_rated])
print('조회수 상위:', [(n['title'], n['views']) for n in top_notices])


### 왜 이 코드가 정답인지

상품과 공지의 반복 단위가 각각 .product-card와 li.notice-item으로 분리되어 있으므로, 먼저 반복 단위를 선택한 다음 내부의 하위 selector를 읽는 방식이 안정적이다. 가격, 재고, 조회수는 쉼표와 단위가 섞인 문자열이므로 clean_int로 숫자만 남겨야 필터링과 정렬이 올바르게 동작한다. href는 상대 경로이므로 urljoin으로 절대 URL 형태를 만들어 저장한다. 최종 산출물은 CSV 파일과 요약 출력이 함께 있으므로 수집 결과를 확인하고 다른 도구로 이어서 분석하기 쉽다.

### 평가 루브릭

- 필수: 상품/공지 개수 일치, CSV 저장, 품절/고평점/조회수 상위 요약.
- 보너스: 카테고리별 평균 가격, 부서별 공지 개수, 요청 윤리 체크리스트 작성.
- 감점: 숫자 변환 누락, selector 오타, 상대 URL 그대로 저장, 실행 순서 의존으로 재실행 불가.

### 추가 채점 메모

이 최종 미션은 "많이 수집하기"가 아니라 "반복 구조를 정확히 읽고 안전하게 저장하기"를 평가한다. 학생 코드가 모범 답안과 달라도 다음 조건을 만족하면 통과시킨다.

| 항목 | 인정 기준 |
|---|---|
| 상품 추출 | 모든 `.product-card` 를 반복하고 필수 필드 6개를 만든다 |
| 공지 추출 | 모든 `li.notice-item` 을 반복하고 필수 필드 4개를 만든다 |
| 숫자 변환 | 가격, 재고, 조회수는 문자열이 아니라 숫자로 비교 가능하다 |
| URL 처리 | 상세 링크는 `urljoin` 또는 동등한 방식으로 절대 URL이 된다 |
| 저장 | CSV가 열렸을 때 헤더와 행 수가 확인된다 |
| 안전 메모 | 실제 사이트 적용 전 확인할 운영 규칙이 들어 있다 |

학생이 `pandas` 로 CSV를 저장해도 결과 파일이 같으면 인정할 수 있다. 다만 1강에서는 표준 라이브러리 `csv` 로도 충분히 가능하다는 점을 한 번 더 설명한다. 학생이 실제 사이트 URL을 임의로 넣어 반복 요청했다면 결과가 맞아도 안전 기준 미달로 보완시킨다.

### 수업 후 확장 질문

빠른 학생에게는 다음 질문을 던진다.

1. 상품 카드가 1,000개라면 print를 어디까지 줄여야 할까?
2. `.price` class 이름이 `.product-price` 로 바뀌면 어떤 부분이 깨질까?
3. CSV를 매일 저장한다면 파일명에 날짜를 넣어야 하는 이유는 무엇일까?
4. 외부 사이트에서 429 상태 코드가 나오면 코드를 어떻게 멈추거나 늦출 수 있을까?

# 레슨 01 — 교사 가이드

## 학습 목표 (학생용보다 더 상세)

학생이 웹 자동화를 단순 크롤링이 아니라 반복 업무를 안전하게 줄이는 과정으로 이해하게 한다. requests와 BeautifulSoup의 역할을 분리해서 설명하고, HTML 구조에서 반복 단위를 찾은 뒤 내부 값을 추출하는 흐름을 반복 훈련한다.

## 2시간 타임라인 (분 단위)

| 시간 | 내용 |
|---:|---|
| 0~10 | 웹 자동화 범위와 윤리 규칙 소개 |
| 10~25 | HTML 구조, 태그/class/속성 설명 |
| 25~40 | 환경 셀 실행, mini_shop.html 읽기 |
| 40~60 | BeautifulSoup selector 실습 |
| 60~75 | 상품 카드 반복 추출 |
| 75~90 | 공지 목록 파싱, 숫자 정리 |
| 90~110 | mission 1~15 풀이 |
| 110~120 | 최종 미션 안내와 제출 기준 정리 |

## 사전 준비물

- Google Colab 접속 가능 계정
- lecture.ipynb, mission.ipynb 링크
- data 폴더의 mini_shop.html, notices.html, targets.csv, robots_sample.txt

## 핵심 개념 강조 포인트

- requests는 HTML 문자열을 가져오는 도구다.
- BeautifulSoup은 HTML 문자열을 탐색 가능한 구조로 바꾸는 도구다.
- select는 여러 개, select_one은 첫 번째 하나를 가져온다.
- text는 화면 글자, attribute는 태그 안 속성값이다.
- 숫자처럼 보여도 HTML에서 읽으면 문자열이다.

## 학생이 자주 막히는 지점과 대처법

- select 결과 리스트에서 바로 text를 읽으려 한다: cards[0]처럼 하나를 먼저 꺼내게 한다.
- class selector 앞의 점을 빼먹는다: .product-card와 product-card 차이를 보여준다.
- 가격 문자열 비교를 한다: 900000과 1500000 비교 전에 타입을 print(type(value))로 확인시킨다.
- href가 상대 경로인 것을 놓친다: urljoin을 쓰는 이유를 브라우저 주소창과 연결해서 설명한다.

## 실습 문제 채점 포인트 (문제별)

1~3번은 환경과 기본 selector 확인, 4~6번은 단일 카드에서 반복 추출로 넘어가는 핵심 구간이다. 7~10번은 딕셔너리 리스트를 기준으로 조건 필터가 가능한지 본다. 11~14번은 다른 HTML 구조와 CSV를 같은 방식으로 읽을 수 있는지 확인한다. 15번은 숫자 결과를 문장으로 바꾸는지 평가한다.

## 최종 미션 평가 루브릭

- 필수 통과: products와 notices가 각각 원본 개수대로 만들어진다.
- 필수 통과: lesson01_products.csv가 생성된다.
- 필수 통과: 품절, 고평점, 조회수 상위 결과가 출력된다.
- 보너스: 카테고리별 가격 평균 또는 부서별 공지 개수를 추가한다.
- 감점: 과도한 외부 요청 코드, 개인정보 수집 예시, 무한 루프 요청 코드.

## 다음 레슨과의 연결고리

2강에서는 한 페이지가 아니라 URL query string과 페이지네이션을 다룬다. 1강에서 만든 '반복 단위 선택 → 내부 값 추출 → 리스트에 저장' 패턴이 그대로 확장된다.

## 수업 중 질문 스크립트

학생이 막혔을 때 바로 selector나 정답 코드를 알려주지 말고 아래 질문 순서로 유도한다.

1. "지금 HTML 문자열은 몇 글자인가요?"
2. "`soup.title` 은 정상적으로 나오나요?"
3. "반복 단위 selector를 넣었을 때 개수는 몇 개인가요?"
4. "첫 번째 카드 하나에서 `.name` 만 먼저 찾아볼까요?"
5. "이 값은 숫자인가요, 문자열인가요? `type()` 으로 확인해 봅시다."
6. "CSV에 저장할 때 열 이름과 딕셔너리 key가 같은가요?"

이 질문들은 학생이 답을 외우는 대신 디버깅 순서를 익히게 한다. 특히 `select` 결과 리스트와 `select_one` 결과 객체를 혼동하는 학생이 많으므로, 카드 하나를 먼저 꺼내는 흐름을 반복해서 보여준다.

## 교사용 사전 실행 체크

수업 전에는 다음을 직접 실행한다.

- [ ] 학생용 통합 노트북을 코랩에서 열고 첫 환경 셀이 성공하는지 확인했다.
- [ ] 선생님용 통합 노트북을 열고 모범 답안 전체를 실행했다.
- [ ] `mini_shop.html` 상품 수와 `notices.html` 공지 수를 알고 있다.
- [ ] `lesson01_products.csv` 생성 후 파일을 열어 헤더와 행을 확인했다.
- [ ] 외부 사이트 요청 예제가 아니라 fixture 기반 실습임을 첫 5분에 안내할 준비가 됐다.

## 문제별 개입 기준

| 구간 | 흔한 상태 | 교사 개입 |
|---|---|---|
| 1~3번 | 파일은 읽었지만 selector가 비어 있음 | HTML 일부를 출력하고 class 이름을 눈으로 찾게 한다 |
| 4~6번 | 첫 카드에서 값 추출 실패 | `first = cards[0]` 이후 한 selector씩 확인시킨다 |
| 7~10번 | products 구조가 흔들림 | `products[0]` 을 출력해 key와 타입을 확인한다 |
| 11~13번 | notices에서 views 정렬 실패 | 조회수를 `clean_int` 로 바꿨는지 확인한다 |
| 14번 | CSV DictReader 사용 실패 | `target_csv.splitlines()[:3]` 로 원문 구조를 보게 한다 |
| 15번 | 요약 문장이 출력과 맞지 않음 | 출력 숫자 하나를 골라 문장에 직접 넣게 한다 |

## 수업 품질 기준

1강은 쉬워 보이지만 뒤 레슨 전체의 습관을 결정한다. 학생이 "돌아가는 코드"만 만들고 끝내면 안 된다. 반드시 다음 세 문장을 말로 설명하게 한다.

- "반복 단위는 `.product-card` 이고, 그 안에서 이름과 가격을 찾았다."
- "가격과 조회수는 문자열이어서 숫자로 변환한 뒤 비교했다."
- "실제 사이트에서는 robots.txt와 요청 간격을 확인한 뒤 실행해야 한다."

이 세 문장을 말할 수 있으면 2강의 페이지네이션에서도 같은 구조를 확장할 수 있다.
